# Machine Learning I: Code-Along (In-Class)

This notebook goes with today's lecture. I lecture and type along on the projector; every time I say **Your turn**, you write the matching code here. You drive a **logistic regression**; I drive a **decision tree**. By the end we have both run the same four steps on the same passengers.

> **The data.** `data/titanic.csv` is the passenger list you read in the pandas review: **891 passengers**, one row each. Today's question: *can we predict who survived from what the passenger list records about each person?*
>
> - `Survived`: 1 = survived, 0 = did not. This is what we predict.
> - `Pclass`: ticket class, 1, 2 or 3
> - `Sex`: `male` or `female`
> - `Fare`: ticket price, in 1912 British pounds
> - `Age`: in years, **missing for 177 passengers** (we leave it out today)

Write code or text everywhere you see *Your answer*. Where a question asks you to notice something, answer in complete sentences and put the numbers in the text, not only in the output. Submit this notebook on Canvas after **Run → Restart Kernel and Run All Cells**; it is graded on completion.

## Part 1: What machine learning is

### Setup

Run this first. It loads the passenger list and the libraries for today.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

titanic = pd.read_csv('data/titanic.csv')
print(titanic.shape)
titanic.head()

### Discussion

**How are statistics and machine learning related? How are they different?** Talk with the person next to you, then write a sentence or two.

*Your answer, in complete sentences:*


### Your turn 1

**Run `.info()` on `titanic`, then compute the fraction of passengers who survived.** Then: if you had no model at all and simply predicted "did not survive" for every passenger, what fraction of the time would you be right?

The mean of a 0/1 column is the fraction of 1s (`.mean()`, from the pandas review). The second number is the **base rate**. Keep it: every accuracy today gets compared with it.

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


## Part 2: The data budget

### Discussion

**When is a good time to split your data: before or after you explore it? Why?**

*Your answer, in complete sentences:*


### Your turn 2

**Split the data so 20% is held out for the test set, with `random_state=123`, and show the first 10 rows of the training data.** Then change the seed a few times and, each time, print the fraction who survived in the **test** set. Does it stay put?

`train_test_split` is from `sklearn.model_selection` (pre-class tutorial, *Splitting data into training and test sets*). Split the whole DataFrame: `train, test = train_test_split(titanic, ...)`. The proportion held out is the `test_size=` argument.

For the seeds: a `for` loop over a short list of seeds. Inside the loop, split again with that seed, then print the mean of the test set's `Survived` column.

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


### Your turn 3

**Split smarter.** Go back to your split from Your turn 2 (20% held out, `random_state=123`) and make it **stratified** by `Survived`. `train_test_split` has an argument for this, `stratify=`: what does it need to be given? Then confirm that the training and test survival rates now match.

*Hint:* the same `.mean()` you used in Your turn 1, once on the training set's `Survived` column and once on the test set's.

In [ ]:
# *Your answer:*


### Your turn 4

**Explore the training set, and only the training set: reproduce the three plots shown in the lecture.** Both functions are from the Seaborn review.

1. **What is the distribution of the outcome, `Survived`?** `sns.catplot()` with `train` as the data, `Survived` on the x-axis, and `kind="count"`: the same `catplot` you used for box and bar plots, with a kind that counts the rows in each category.
2. **Does `Fare` look different for survivors and non-survivors?** `sns.displot()` with `Fare` on the x-axis and `col='Survived'` to get one panel per outcome (the Seaborn review split the fMRI signal into panels by event the same way). Try a few values of `bins=`.
3. **How does the survival rate differ across `Pclass` and `Sex`?** `sns.catplot()` with `kind="bar"`, `Pclass` on the x-axis, `Survived` on the y-axis, and `hue='Sex'` (the Seaborn review's bar plot of bills by day, with smoker as the hue). Each bar's height is the mean of `Survived` for that group, and the mean of a 0/1 column is the survival rate.

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


### Your turn 5

**Make the data model-ready.**

1. Add a 0/1 column `Female` to `titanic`: `(titanic['Sex'] == 'female').astype(int)` (comparison on a column, then `.astype(int)` from Data Wrangling I).
2. Redo your stratified split from Your turn 3, so both halves get the new column.
3. Build the feature table and the labels: `features = ['Pclass', 'Fare', 'Female']`, then `X_train = train[features]` and `y_train = train['Survived']`. Print `X_train.shape`. Leave `test` alone: it stays locked away until the final test.

In [ ]:
# *Your answer:*


## Part 3: Fitting a model

### Your turn 6

**Create a logistic regression model and fit it to the training data.** Call it `logi`.

The model is `LogisticRegression`, from `sklearn.linear_model`. Same two verbs as the tree on the screen: create, then `.fit(X_train, y_train)`.

In [ ]:
# *Your answer:*


### Your turn 7

**Predict.** Do for `logi` what the lecture did for the tree: get its hard predictions (`.predict(X_train)`) and its probabilities (`.predict_proba(X_train)`), attach them to `train` with `.assign(pred=..., prob_survived=..., prob_died=...)`, and show the first eight rows of `pred`, `prob_survived`, `prob_died`, `Survived`, `Pclass`, `Fare`, `Female`, `Name`.

In [ ]:
# *Your answer:*


### Your turn 8

**Look inside the model.** Logistic regression can do something the tree cannot: its coefficients tell you how each feature moves the odds. Run this:

```python
pd.Series(np.exp(logi.coef_[0]), index=features).round(2)
```

`np.exp()` turns the log-odds coefficients into **odds ratios**. In a sentence each: what does being female do to the odds of surviving? What does each step down in class do?

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


## Part 4: Metrics for model performance

Everything in this part is computed on the **training** set. Thursday is about why that is not enough.

The function below packages the three metrics from the lecture into one call. Run it; you will use it on every model from here on. (Functions: the Ifs, Loops and Functions review. `.ravel()` flattens the 2×2 confusion matrix into its four cells in reading order: TN, FP, FN, TP.)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report

def classification_summary(y_true, y_pred):
    """Accuracy, sensitivity and specificity from a confusion matrix, as a Series."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return pd.Series({'accuracy':    (tp + tn) / (tp + tn + fp + fn),
                      'sensitivity': tp / (tp + fn),
                      'specificity': tn / (tn + fp)})

### Your turn 9

**Compute the confusion matrix for `logi` and its accuracy, sensitivity and specificity.** Use `ConfusionMatrixDisplay.from_estimator(logi, X_train, y_train, display_labels=['died', 'survived'])` for the picture, then `classification_summary(y_train, logi.predict(X_train))`.

Compared with the tree on the screen, which kind of mistake does your model make more of: calling a survivor "died", or calling a non-survivor "survived"? Does its accuracy beat the base rate from Your turn 1?

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


### Your turn 10

**Move the threshold.** `.predict()` calls a passenger a survivor when the probability is at least 0.5. Pick a different threshold (anything but 0.5), turn the probabilities into 0/1 predictions with it, and run `classification_summary` on those.

Pattern: `prob = logi.predict_proba(X_train)[:, 1]`, then `pred = (prob >= your_threshold).astype(int)`.

In a sentence: which mistake did you make more of at your threshold, and when might a person *want* that trade?

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


### Your turn 11

**Plot the ROC curve for `logi` and compute its AUC.** `RocCurveDisplay` and `roc_auc_score` are in `sklearn.metrics`. The AUC needs the probabilities, not the hard calls: `roc_auc_score(y_train, logi.predict_proba(X_train)[:, 1])`.

To put the tree and your model on one plot, draw one, keep its axes, and pass them to the other: `disp = RocCurveDisplay.from_estimator(logi, X_train, y_train, name='logistic')`, then a second `from_estimator(...)` with `ax=disp.ax_`. (If you did not build the tree, plot `logi` alone; that is fine.)

In [ ]:
# *Your answer:*


### Your turn 12 (if time allows)

**One line away.** The pre-class tutorial's model was `KNeighborsClassifier(n_neighbors=3)` (from `sklearn.neighbors`). Create it, fit it, and run `classification_summary` and `roc_auc_score` on it. Three models, one grammar. Which looks best on the training data?

Keep your answer. On Thursday we ask whether the best-looking one is the one to trust.

In [ ]:
# *Your answer:*


*Your answer, in complete sentences:*


---

Every number you computed today came from passengers the model had already seen. On Thursday: the test set comes out of the vault, and **cross-validation** tells us which of today's numbers to believe.